In [ ]:
import nest_asyncio
# Jupyter Notebook에서 비동기 코드 실행을 위한 라이브러리
# 일반 Python 파일에서는 필요없지만, Jupyter에서는 필수입니다.
nest_asyncio.apply()

import asyncio  # 비동기 프로그래밍을 위한 Python 내장 라이브러리
import logging  # 프로그램 실행 과정을 기록하기 위한 로깅 라이브러리
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
# Playwright: 웹브라우저를 자동화하는 라이브러리 (Chrome, Firefox 등 제어 가능)
# TimeoutError: 웹페이지 로딩이 너무 오래 걸릴 때 발생하는 에러
from playwright.async_api import BrowserContext
import pandas as pd  # 데이터를 표 형태로 다루기 위한 라이브러리
from datetime import datetime  # 날짜와 시간을 다루기 위한 라이브러리
import re  # 정규표현식 - 텍스트 패턴 매칭을 위한 라이브러리
from typing import List, Dict, Optional  # 타입 힌팅용 - 코드 가독성 향상  / 타입 힌팅: 파이썬에서 함수의 파라미터와 반환 값의 타입을 명시적으로 지정하는 것을 의미
from dataclasses import dataclass  # 클래스를 쉽게 만들어주는 데코레이터


# ===== 로깅 설정 =====
# 프로그램 실행 과정을 콘솔에 출력하기 위한 설정
logging.basicConfig(level=logging.INFO)  # INFO 레벨 이상의 메시지만 출력
logger = logging.getLogger(__name__)  # 현재 모듈용 로거 생성

@dataclass # 데코레이터 - __init__, __repr__, __eq__ 등 자동 생성
class CrawlConfig:
    """
    크롤링 작업의 모든 설정값을 관리하는 클래스
    
    각 설정값의 의미:
    - base_url: 크롤링할 웹사이트의 기본 주소
    - max_concurrent: 동시에 실행할 수 있는 최대 작업 수 (너무 높으면 서버에 부담)
    - timeout: 웹페이지 로딩을 기다리는 최대 시간 (밀리초)
    - max_retries: 실패했을 때 재시도할 최대 횟수
    - batch_size: 한 번에 처리할 데이터 개수 (메모리 관리용)
    """
    base_url: str = "https://www.aitimes.com"  # AITIMES 웹사이트 주소
    max_concurrent: int = 5  # 동시 작업 5개까지 허용
    timeout: int = 30000  # 30초 타임아웃
    max_retries: int = 3  # 최대 3번 재시도
    batch_size: int = 50  # 한 번에 50개씩 처리
    
class RobustCrawler:
    """
    안정적이고 효율적인 웹 크롤링을 수행하는 클래스
    
    주요 기능:
    1. 동시성 제어 (서버 부하 방지)
    2. 자동 재시도 (네트워크 오류 대응)
    3. 데이터 품질 검증
    4. 배치 처리 (대용량 데이터 처리)
    """
    def __init__(self, config: CrawlConfig = None):
        """
        크롤러 초기화
        
        Args:
            config: 크롤링 설정 (없으면 기본값 사용)
        """
        self.config = config or CrawlConfig()  # 설정이 없으면 기본 설정 사용
        # Semaphore: 동시 실행 작업 수를 제한하는 도구
        self.sem = asyncio.Semaphore(self.config.max_concurrent)
        # 크롤링 통계 정보
        self.stats = {"success": 0, "failed": 0, "retried": 0}
    
    async def fetch_with_retry(self, context: BrowserContext, url: str, 
                             operation_name: str = "fetch") -> Optional[str]:
        """
        웹페이지를 가져오는 핵심 함수 (재시도 로직 포함)
        
        이 함수가 중요한 이유:
        - 네트워크는 불안정할 수 있음 (일시적 연결 끊김, 서버 과부하 등)
        - 한 번 실패해도 재시도하면 성공할 가능성이 높음
        - 지수 백오프로 서버 부하를 줄임
        
        Args:
            context: 브라우저 컨텍스트 (쿠키, 세션 등 유지)
            url: 가져올 웹페이지 주소
            operation_name: 작업 이름 (로그용)
            
        Returns:
            성공시 페이지 객체, 실패시 None
        """
        for attempt in range(self.config.max_retries):
            async with self.sem:  # 동시성 제어: 너무 많은 요청이 한 번에 가지 않도록 제한
                page = None
                try:
                    page = await context.new_page() # 새 페이지 탭 생성
                    response = await page.goto(url, timeout=self.config.timeout) # 웹페이지로 이동 (타임아웃 설정)
                    
                    if response.status == 200: # 성공
                        self.stats["success"] += 1
                        return page
                    elif response.status in [429, 503, 502]:  # 재시도 가능한 에러 - 429: 너무 많은 요청, 503: 서비스 불가, 502: 게이트웨이 에러
                        raise Exception(f"Server error: {response.status}")
                    else: # 기타 에러 (404 등)
                        logger.warning(f"HTTP {response.status} for {url}")
                        return None
                        
                except (PlaywrightTimeoutError, Exception) as e:
                    if attempt == self.config.max_retries - 1: # 마지막 시도에서도 실패하면 완전히 포기
                        logger.error(f"{operation_name} 최종 실패 {url}: {str(e)}")
                        self.stats["failed"] += 1
                        return None
                    
                    # 지수 백오프: 재시도할 때마다 대기 시간을 늘림 - 첫 번째 재시도: 0.5초, 두 번째: 1초, 세 번째: 2초... 이런 식으로 증가
                    delay = 0.5 * (2 ** attempt)
                    logger.info(f"{operation_name} 재시도 {attempt + 1}/{self.config.max_retries} "
                              f"({delay}초 후): {url}")
                    self.stats["retried"] += 1
                    
                    # 실패한 페이지는 메모리에서 해제
                    if page:
                        await page.close()
                    await asyncio.sleep(delay) # 지정된 시간만큼 대기 후 재시도
                    continue
                    
        return None


    def contains_email(self, text: str) -> bool:
        """이메일 포함 여부 확인"""
        pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
        return bool(re.search(pattern, text))


    def convert_to_ymd(self, date_str: str) -> str:
        """날짜 형식 변환"""
        try:
            date = date_str.split()[0]
            if len(date) < 6:
                this_year = datetime.now().year
                date_obj = datetime.strptime(f"{this_year}.{date}", "%Y.%m.%d")
            else:
                date_obj = datetime.strptime(f"{date}", "%Y.%m.%d")
            return date_obj.strftime("%Y-%m-%d")

        except ValueError:
            return datetime(9999, 1, 1).strftime("%Y-%m-%d")


    def clean_text(self, text: str) -> str:
        """텍스트 정제"""
        text = text.replace('(', ' ').replace(')', ' ')
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'[^\w\s가-힣.,]', '', text)
        return text


    def validate_article_data(self, article: Dict) -> bool:
        """
        수집된 기사 데이터의 품질을 검증하는 함수
        
        데이터 품질이 중요한 이유:
        - 제목이 너무 짧으면 의미있는 기사가 아닐 가능성
        - 본문이 너무 짧으면 광고나 공지사항일 가능성
        - 날짜가 없으면 시계열 분석 불가
        
        Args:
            article: 검증할 기사 데이터 딕셔너리
            
        Returns:
            품질 기준을 만족하면 True, 아니면 False
        """
        if not article.get('title') or len(article['title']) < 5: # 제목이 너무 짧으면 의미있는 기사가 아닐 가능성
            return False
        if not article.get('body') or len(article['body']) < 30: # 본문이 너무 짧으면 광고나 공지사항일 가능성
            return False
        if not article.get('date'): # 날짜가 없으면 시계열 분석 불가
            return False
        return True


    async def fetch_article_content(self, context: BrowserContext, url: str) -> str:
        """
        개별 기사의 본문 내용을 추출하는 함수
        
        AITIMES의 기사 본문은 특정 CSS 클래스에 있습니다.
        이를 찾아서 텍스트만 깔끔하게 추출합니다.
        
        Args:
            context: 브라우저 컨텍스트
            url: 기사 상세 페이지 URL
            
        Returns:
            추출된 기사 본문 (실패시 빈 문자열)
        """
        page = await self.fetch_with_retry(context, url, "기사 본문")
        if not page:
            return ""
            
        try:
            articles = await page.query_selector_all('.article-veiw-body.view-page.font-size17 > p')
            body_text = ""
            
            for article in articles:
                text = await article.inner_text()
                if self.contains_email(text): # 이메일이 포함된 문단은 제외 (기자 정보 등)
                    continue
                body_text += self.clean_text(text) + ' '  # 텍스트 정제 후 추가
            
            return body_text.strip()
            
        except Exception as e:
            logger.error(f"기사 본문 추출 실패 {url}: {str(e)}")
            return ""
        finally:
            await page.close()


    async def fetch_page_articles(self, context: BrowserContext, page_num: int) -> List[Dict]:
        """
        목록 페이지에서 기사들의 메타데이터(제목, 링크, 작성자, 날짜)를 추출
        
        AITIMES는 페이지별로 기사 목록을 보여줍니다.
        각 페이지에서 기사의 기본 정보를 먼저 수집합니다.
        
        Args:
            context: 브라우저 컨텍스트
            page_num: 페이지 번호
            
        Returns:
            기사 메타데이터 리스트
        """
        page_url = f"{self.config.base_url}/news/articleList.html?page={page_num}" # 페이지 URL 생성
        page = await self.fetch_with_retry(context, page_url, f"페이지 {page_num}") # 페이지 가져오기 (재시도 로직 포함)
        
        if not page: # 페이지 가져오기 실패 시 빈 리스트 반환
            return []
            
        try:
            articles = await page.query_selector_all('ul.type1 > li') # 기사 목록 추출
            article_meta = [] # 기사 메타데이터 저장 리스트
            
            for article in articles: # 각 기사에 대해 반복
                try:
                    a_tag = await article.query_selector('h4.titles > a') # 제목과 링크 추출
                    if not a_tag: # 링크가 없으면 건너뜀
                        continue
                        
                    title = await a_tag.inner_text() # 제목 추출
                    link = self.config.base_url + await a_tag.get_attribute('href') # 링크 추출
                    
                    author_tag = await article.query_selector('em.info.name') # 작성자 추출
                    author = await author_tag.inner_text() if author_tag else "Unknown"
                    
                    date_tag = await article.query_selector('em.info.dated') # 날짜 추출
                    date = self.convert_to_ymd(await date_tag.inner_text()) if date_tag else "" # 날짜 형식 변환

                    article_meta.append({
                        'title': self.clean_text(title), # 정제된 제목
                        'link': link.strip(), # 정제된 링크
                        'author': author.strip(), # 정제된 작성자
                        'date': date # 정제된 날짜
                    })
                    
                except Exception as e:
                    logger.warning(f"개별 기사 메타데이터 추출 실패: {str(e)}")
                    continue
            
            logger.info(f"페이지 {page_num}: {len(article_meta)}개 기사 발견")
            return article_meta
            
        except Exception as e:
            logger.error(f"페이지 {page_num} 전체 실패: {str(e)}")
            return []
        finally:
            await page.close()


    async def process_articles_batch(self, context: BrowserContext, 
                                   articles_meta: List[Dict]) -> List[Dict]:
        """
        기사 메타데이터 리스트를 받아서 각 기사의 본문을 동시에 가져오는 함수
        
        비동기 프로그래밍의 핵심:
        - 기사 100개의 본문을 하나씩 가져오면 매우 느림
        - 동시에 여러 개를 가져오면 훨씬 빠름 (단, 서버 부하 고려 필요)
        
        Args:
            context: 브라우저 컨텍스트
            articles_meta: 기사 메타데이터 리스트
            
        Returns:
            본문이 포함된 완전한 기사 데이터 리스트
        """
        tasks = [] # 본문 추출 작업 목록
        for meta in articles_meta:
            task = self.fetch_article_content(context, meta['link']) # 각 기사의 본문 추출
            tasks.append(task) # 작업 목록에 추가
        
        bodies = await asyncio.gather(*tasks, return_exceptions=True) # 모든 작업 동시 실행
        
        results = [] # 최종 결과 저장 리스트
        for meta, body in zip(articles_meta, bodies): # 각 기사의 메타데이터와 본문 쌍을 순회
            if isinstance(body, Exception):
                logger.warning(f"기사 본문 추출 실패: {meta['link']}")
                body = ""
            
            article_data = {**meta, 'body': body} # 메타데이터와 본문 결합
            if self.validate_article_data(article_data): # 품질 검증
                results.append(article_data) # 검증된 데이터 저장
            else: # 품질 검증 실패
                logger.info(f"품질 검증 실패: {meta['title'][:30]}...")
        
        return results # 최종 결과 반환


    async def crawl_optimized(self, total_pages: int) -> pd.DataFrame:
        """
        최적화된 크롤링을 수행하는 메인 함수
        
        이 함수는 전체 크롤링 프로세스를 관리합니다:
        1. 브라우저 설정 및 실행
        2. 데이터 규모에 따른 처리 방식 선택
        3. 결과 검증 및 통계 출력
        4. 리소스 정리
        
        Args:
            total_pages: 크롤링할 총 페이지 수
            
        Returns:
            수집된 기사 데이터가 담긴 pandas DataFrame
        """
        
        async with async_playwright() as p:
            # 브라우저 최적화 설정

            browser = await p.chromium.launch(
                headless=True,
                args=['--no-sandbox', '--disable-dev-shm-usage', '--disable-images']
            )
            """
            headless=True
            브라우저 창을 화면에 표시하지 않음, 백그라운드에서 실행 (서버 환경에 적합), 메모리와 CPU 사용량 절약

            --no-sandbox
            Chrome의 보안 샌드박스 비활성화, Docker나 제한된 환경에서 필요, 보안이 약해지지만 호환성 향상

            --disable-dev-shm-usage
            /dev/shm 파티션 사용 비활성화, 메모리 부족 환경에서 Chrome 크래시 방지, Docker 컨테이너에서 특히 유용

            --disable-images
            이미지 로딩 완전 차단, 페이지 로딩 속도 대폭 향상,네트워크 사용량 절약
            """
            
            context = await browser.new_context(
                user_agent='Mozilla/5.0 (compatible; WebCrawler/1.0)',
                extra_http_headers={'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8'}
            )
            """
            user_agent
            브라우저 식별 정보 웹서버에게 "크롤러임"을 정직하게 알림, 일부 사이트는 봇 차단, 일부는 허용

            Accept-Language 헤더
            ko-KR,ko;q=0.9,en;q=0.8 분석: 
                ko-KR: 한국어(한국) 최우선 (q=1.0 기본값), ko;q=0.9: 일반 한국어 차순위 (90% 선호도), en;q=0.8: 영어 최후순위 (80% 선호도)
            웹사이트가 한국어 콘텐츠를 우선 제공하도록 요청
            """
            
            try:
                logger.info(f"크롤링 시작: {total_pages}페이지")
                all_articles = []

                
                # 대용량 데이터인 경우 배치 처리 적용
                if total_pages > 20:  # 검증 결과 기반 임계값
                    logger.info("대용량 데이터 감지 - 배치 처리 모드")
                    all_articles = await self._batch_process(context, total_pages)
                else:
                    logger.info("소규모 데이터 - 직접 처리 모드")
                    all_articles = await self._direct_process(context, total_pages)
                
                # 최종 검증 및 통계
                valid_articles = [a for a in all_articles if self.validate_article_data(a)]
                
                logger.info(f"""크롤링 완료:
                총 수집: {len(all_articles)}개
                유효 데이터: {len(valid_articles)}개
                성공률: {len(valid_articles)/len(all_articles)*100:.1f}%
                재시도: {self.stats['retried']}회
                실패: {self.stats['failed']}회""")
                
                return pd.DataFrame(valid_articles)
                
            finally:
                await context.close()
                await browser.close()


    async def _direct_process(self, context: BrowserContext, total_pages: int) -> List[Dict]:
        """
        소규모 데이터를 위한 직접 처리 방식
        
        단계:
        1. 모든 페이지에서 메타데이터 동시 수집
        2. 수집된 메타데이터로 본문 동시 수집
        
        Args:
            context: 브라우저 컨텍스트
            total_pages: 총 페이지 수
            
        Returns:
            완전한 기사 데이터 리스트
        """
        # 1단계: 모든 페이지 메타데이터 수집
        page_tasks = [self.fetch_page_articles(context, i) for i in range(1, total_pages + 1)]
        page_results = await asyncio.gather(*page_tasks)
        
        # 평면화
        all_meta = [item for sublist in page_results for item in sublist]
        logger.info(f"메타데이터 수집 완료: {len(all_meta)}개")
        
        # 2단계: 본문 수집
        return await self.process_articles_batch(context, all_meta)

    async def _batch_process(self, context: BrowserContext, total_pages: int) -> List[Dict]:
        """
        대용량 데이터를 위한 배치 처리 방식
        
        메모리 효율성을 위해 전체 페이지를 작은 배치로 나누어 처리
        예: 100페이지 → 50페이지씩 2번 처리
        
        Args:
            context: 브라우저 컨텍스트
            total_pages: 총 페이지 수
            
        Returns:
            완전한 기사 데이터 리스트
        """
        all_articles = []
        batch_size = self.config.batch_size
        
        # 전체 페이지를 배치 크기로 나누어 처리
        for i in range(0, total_pages, batch_size):
            batch_end = min(i + batch_size, total_pages)
            batch_pages = list(range(i + 1, batch_end + 1))
            
            logger.info(f"배치 {i//batch_size + 1} 처리 중: 페이지 {batch_pages[0]}-{batch_pages[-1]}")
            
            # 배치 내 페이지 메타데이터 수집
            page_tasks = [self.fetch_page_articles(context, page_num) for page_num in batch_pages]
            page_results = await asyncio.gather(*page_tasks)
            
            # 배치 내 본문 수집
            batch_meta = [item for sublist in page_results for item in sublist]
            batch_articles = await self.process_articles_batch(context, batch_meta)
            
            all_articles.extend(batch_articles)
            logger.info(f"배치 {i//batch_size + 1} 완료: {len(batch_articles)}개 수집")
            
            # 메모리 정리 (검증에서 효과 확인)
            del page_results, batch_meta, batch_articles
            await asyncio.sleep(0.1)  # 시스템 안정성을 위한 짧은 휴식
        
        return all_articles


# 사용 예시
async def main():
    """최적화된 크롤러 실행"""
    
    # 설정 - 환경에 맞게 조정
    config = CrawlConfig(
        max_concurrent=3,  # 안정성을 위해 보수적 설정
        timeout=30000,
        max_retries=3,
        batch_size=30      # 대용량 시에만 효과
    )
    
    crawler = RobustCrawler(config)
    
    try:
        df = await crawler.crawl_optimized(1200)
        
        if not df.empty:
            df.to_csv('crawled_data.csv', index=False, encoding='utf-8-sig') # CSV 파일로 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용)
            print(f"크롤링 성공: {len(df)}개 기사 저장")
            print(df.head())
        else:
            print("수집된 데이터가 없습니다.")

    except Exception as e:
        logger.error(f"크롤링 실패: {str(e)}")

if __name__ == "__main__":
    asyncio.run(main())

INFO:__main__:크롤링 시작: 1200페이지
INFO:__main__:대용량 데이터 감지 - 배치 처리 모드
INFO:__main__:배치 1 처리 중: 페이지 1-30
INFO:__main__:페이지 1: 20개 기사 발견
INFO:__main__:페이지 3: 20개 기사 발견
INFO:__main__:페이지 2: 20개 기사 발견
INFO:__main__:페이지 4: 20개 기사 발견
INFO:__main__:페이지 5: 20개 기사 발견
INFO:__main__:페이지 6: 20개 기사 발견
INFO:__main__:페이지 7: 20개 기사 발견
INFO:__main__:페이지 8: 20개 기사 발견
INFO:__main__:페이지 9: 20개 기사 발견
INFO:__main__:페이지 10: 20개 기사 발견
INFO:__main__:페이지 11: 20개 기사 발견
INFO:__main__:페이지 12: 20개 기사 발견
INFO:__main__:페이지 13: 20개 기사 발견
INFO:__main__:페이지 14: 20개 기사 발견
INFO:__main__:페이지 15: 20개 기사 발견
INFO:__main__:페이지 16: 20개 기사 발견
INFO:__main__:페이지 17: 20개 기사 발견
INFO:__main__:페이지 18: 20개 기사 발견
INFO:__main__:페이지 19: 20개 기사 발견
INFO:__main__:페이지 20: 20개 기사 발견
INFO:__main__:페이지 21: 20개 기사 발견
INFO:__main__:페이지 22: 20개 기사 발견
INFO:__main__:페이지 23: 20개 기사 발견
INFO:__main__:페이지 24: 20개 기사 발견
INFO:__main__:페이지 26: 20개 기사 발견
INFO:__main__:페이지 27: 20개 기사 발견
INFO:__main__:페이지 25: 20개 기사 발견
INFO:__main__:페이지 29: 20개 기사 발견
INFO:__main__

크롤링 성공: 20716개 기사 저장
                                       title  \
0          허깅페이스, 맥북에서 구동하는 로봇용 시각언어행동 모델 출시   
1             레딧, 무단으로 AI 훈련 데이터 사용한 앤트로픽 고소   
2  미스트랄, 코딩 AI 역량 총 집결한 기업용 어시스턴트 미스트랄 코드 출시   
3  오픈AI, 업무용 AI 도구 대거 출시...MS구글과 에이전트 경쟁 불가피   
4                  미중 갈등으로 애플 인텔리전스 중국 출시 지연   

                                                link  author        date  \
0  https://www.aitimes.com/news/articleView.html?...   박찬 기자  2025-06-05   
1  https://www.aitimes.com/news/articleView.html?...   박찬 기자  2025-06-05   
2  https://www.aitimes.com/news/articleView.html?...   박찬 기자  2025-06-05   
3  https://www.aitimes.com/news/articleView.html?...   박찬 기자  2025-06-05   
4  https://www.aitimes.com/news/articleView.html?...  임대준 기자  2025-06-05   

                                                body  
0  허깅페이스가 범용 로봇 에이전트 개발을 위한 오픈소스 비전언어행동 VLA 모델을 선...  
1  소셜 미디어 플랫폼 레딧이 앤트로픽을 상대로 동의 없이 게시물을 수집해 인공지능 A...  
2  미스트랄 AI가 기업을 위한 인공지능 AI 코딩 어시스턴트를 공개했다. 이번 출시는...  
3  오픈AI가 마이크로